# Can I afford to live here? Housing affordability in Amsterdam

**Question:** Which Amsterdam districts are most and least affordable, and what does an average home cost in years of a PhD salary?

**Data:** Statistics Netherlands (CBS), *Kerncijfers wijken en buurten* (open data, CC BY 4.0).

This notebook walks through the analysis step by step. `run.py` does the same in one command.

In [ ]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from amsterdam_housing import data, analysis, plots, geo

## 1. Find the CBS tables
CBS publishes one *Kerncijfers wijken en buurten* table per year. We list them all.

In [ ]:
tables = data.find_kwb_tables()
tables

## 2. Download one year
The first download takes a minute; after that the table is cached in `data/raw/`.
Income figures arrive later than housing figures, so if the newest year has many empty income values, try the year before.

In [ ]:
YEAR = list(tables)[0]   # newest year; change to e.g. 2023 if income is missing
raw = data.download_table(tables[YEAR])
raw.shape

In [ ]:
# The column names are in Dutch and end with a number that changes per year
[c for c in raw.columns if "WOZ" in c or "Inkomen" in c]

## 3. Keep Amsterdam districts and clean

In [ ]:
districts = data.clean_amsterdam(raw, level="Wijk")
print(len(districts), "districts;", f"{data.coverage(districts):.0%} have both home value and income")
districts.head()

## 4. Calculate affordability
- **price_to_income** = average home value ÷ average income per income recipient (higher = less affordable)
- **years_phd_salary** = average home value ÷ first-year UvA PhD gross salary

In [ ]:
result = analysis.add_affordability(districts)
city = data.city_reference(raw)
city_ratio = city["home_value"] / city["income"]
print(f"Amsterdam average price-to-income: {city_ratio:.1f}")
result[["name", "home_value", "income", "price_to_income", "years_phd_salary"]].head(10)

## 5. Charts

In [ ]:
from IPython.display import Image, display
for path in [plots.top_bottom_bar(result, city_ratio, YEAR),
             plots.income_vs_value(result, YEAR),
             plots.phd_years(result, YEAR)]:
    display(Image(filename=str(path)))

## 6. Map

In [ ]:
shapes = geo.load_district_shapes(YEAR)
if shapes is not None:
    display(Image(filename=str(plots.affordability_map(result, shapes, YEAR))))

## 7. Findings
Write your own three findings here after looking at the charts.

In [ ]:
print(analysis.summarise(result, city, YEAR))